# SynBioCrow 2.2.0 — published-artifact acceptance
Final acceptance of the **published GitHub v2.2.0 assets**. A green run closes the 2.2 release lifecycle and produces archived acceptance evidence.


In [ ]:
import os,sys,json,hashlib,subprocess,shutil,zipfile,urllib.request
from pathlib import Path
from google.colab import drive
OWNER="theiman112860"; REPO="SynBioCrow"; TAG="v2.2.0"
EXPECTED_COMMIT="1e54f885120209746a385443b9390590d442dc4c"
ZENODO_DOI="10.5281/zenodo.23027052"
ASSETS={
"SynBioCrow_2.2.0_PACKAGE_SUMMARY.json":"0e568022534abd60014ae389e1be6aad32e0b464408a7d15942d15854fe05302",
"SynBioCrow_2.2.0_release_manifest.json":"62cbfaa845ac4255e6a9f33f2180292d4155b875c2ba3400650db31c4ec94f71",
"SynBioCrow_2.2.0_SOURCE_REPRODUCIBILITY.zip":"413c704a6446a503ba982f5312093f96877db7be7c1c6e8aebf11ec51913a1c6"}
ROOT=Path("/content/synbiocrow_2_2_0_acceptance")
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir()
def sha256(p):
 h=hashlib.sha256()
 with open(p,"rb") as f:
  for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
 return h.hexdigest()
def run(cmd,cwd=None,timeout=1800):
 print("$"," ".join(map(str,cmd)),flush=True)
 p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
 print(p.stdout,flush=True)
 if p.returncode: raise RuntimeError(f"command failed rc={p.returncode}: {cmd}")
 return p.stdout
print("Final acceptance target",TAG,EXPECTED_COMMIT,ZENODO_DOI)


In [ ]:
for name,expected in ASSETS.items():
 url=f"https://github.com/{OWNER}/{REPO}/releases/download/{TAG}/{name}"
 dst=ROOT/name
 urllib.request.urlretrieve(url,dst)
 got=sha256(dst)
 print(name,dst.stat().st_size,got)
 assert got==expected,(name,got,expected)
print("FINAL PUBLISHED ASSET DIGESTS: PASS")


In [ ]:
summary=json.loads((ROOT/"SynBioCrow_2.2.0_PACKAGE_SUMMARY.json").read_text())
manifest=json.loads((ROOT/"SynBioCrow_2.2.0_release_manifest.json").read_text())
assert summary["git_commit"]==EXPECTED_COMMIT,summary["git_commit"]
assert manifest["git_commit"]==EXPECTED_COMMIT,manifest["git_commit"]
assert manifest["release"]=="2.2.0"
assert manifest["release_kind"]=="final"
zip_path=ROOT/summary["source_zip"]
assert sha256(zip_path)==summary["source_zip_sha256"]
src=ROOT/"source";src.mkdir()
with zipfile.ZipFile(zip_path) as z:z.extractall(src)
bad=[]
for item in manifest["files"]:
 p=src/item["path"]
 if not p.is_file() or p.stat().st_size!=item["bytes"] or sha256(p)!=item["sha256"]: bad.append(item["path"])
assert not bad,bad[:20]
print("FINAL PACKAGE/TAG COMMIT MATCH: PASS")
print("FINAL M9 INVENTORY: PASS",len(manifest["files"]),"files")


In [ ]:
site=ROOT/"isolated_site";site.mkdir()
run([sys.executable,"-m","pip","install","--upgrade","--target",site,str(src)])
def iso(code,cwd=None,timeout=1800):
 return run([sys.executable,"-I","-c",f"import sys;sys.path.insert(0,{str(site)!r});\n"+code],cwd=cwd,timeout=timeout)
iso("import synbiocrow;print('isolated import PASS',synbiocrow.__file__)")
iso("from importlib.metadata import version;v=version('synbiocrow');print(v);assert v=='2.2.0'")
print("FINAL CLEAN INSTALL/VERSION: PASS")


In [ ]:
run([sys.executable,"-m","pip","install","--upgrade","--target",site,"pytest"])
tests=src/"tests";assert tests.is_dir()
iso(f"import pytest;raise SystemExit(pytest.main(['-q',{str(tests)!r}]))",cwd=src)
print("FINAL PUBLISHED TEST SUITE: PASS")


In [ ]:
iso(f"import runpy,sys;sys.argv=['m7_release_readiness.py'];runpy.run_path({str(src/'scripts/m7_release_readiness.py')!r},run_name='__main__')",cwd=src)
iso(f"import runpy,sys;sys.argv=['build_m9_release.py','--check'];runpy.run_path({str(src/'scripts/build_m9_release.py')!r},run_name='__main__')",cwd=src)
print("FINAL PUBLISHED M7 + M9: PASS")


In [ ]:
cert=(src/"docs/RETROPATH_STANDALONE_EQUIVALENCE.md").read_text()
for x in ["EXACT_COMPOUND_TRANSITION_MULTISET","historical rows: 34","standalone rows: 34","precision: 1.0","recall: 1.0","Jaccard: 1.0","KNIME is not required"]:
 assert x.lower() in cert.lower(),x
print("RETROPATH CERTIFICATE: PASS")
# Verify the archived Zenodo DOI resolves.
with urllib.request.urlopen("https://doi.org/"+ZENODO_DOI,timeout=60) as resp:
 print("Zenodo DOI resolved",resp.status,resp.geturl())
print("ZENODO DOI: PASS",ZENODO_DOI)


In [ ]:
report={"schema":"synbiocrow.final_acceptance.v1","release":"2.2.0","tag":TAG,"commit":EXPECTED_COMMIT,"zenodo_doi":ZENODO_DOI,
"published_asset_sha256":ASSETS,"source_zip_sha256":sha256(zip_path),"source_zip_bytes":zip_path.stat().st_size,
"manifest_file_count":manifest["file_count"],"m7_reproducibility_digest":manifest["m7_reproducibility_digest"],
"checks":{"asset_digests":"PASS","package_tag_commit_match":"PASS","manifest_inventory":"PASS","clean_install":"PASS","tests":"PASS","m7":"PASS","m9":"PASS","retropath_certificate":"PASS","zenodo_doi":"PASS"},"acceptance":"PASS","release_lifecycle":"CLOSED"}
rp=ROOT/"SynBioCrow_2.2.0_FINAL_ACCEPTANCE_REPORT.json";rp.write_text(json.dumps(report,indent=2,sort_keys=True)+"\n")
bundle=ROOT/"SynBioCrow_2.2.0_FINAL_ACCEPTANCE_BUNDLE.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
 for p in [rp,ROOT/"SynBioCrow_2.2.0_PACKAGE_SUMMARY.json",ROOT/"SynBioCrow_2.2.0_release_manifest.json"]:z.write(p,p.name)
print(json.dumps(report,indent=2))
print("report sha256",sha256(rp));print("bundle sha256",sha256(bundle))
drive.mount("/content/drive",force_remount=False)
out=Path("/content/drive/MyDrive/SynBioCrow/release/2.2.0/acceptance");out.mkdir(parents=True,exist_ok=True)
for p in [rp,bundle]:shutil.copy2(p,out/p.name);print("saved",out/p.name)
print("SYNBIOCROW 2.2.0 PUBLISHED-ARTIFACT ACCEPTANCE: PASS")
print("2.2 RELEASE LIFECYCLE: CLOSED")
